# Flatmate RL GRPO Step-2 Curriculum

This notebook is a minimal GRPO starter for `flatmate_rl`.
It only trains the first two workflow steps:

1. ask for the missing buyer details
2. store the buyer profile

The goal is to keep the reward simple enough to bootstrap the broker policy before training on later booking steps.

In [ ]:
%pip install -q trl transformers accelerate datasets peft bitsandbytes sentencepiece

from __future__ import annotations

import json
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from datasets import Dataset
from flatmate_rl import FlatmateRlAction
from flatmate_rl.server.flatmate_rl_environment import FlatmateRlEnvironment
from flatmate_rl.server.heuristic_policy import expected_policy_action

print('imports ready')

In [ ]:
TARGET_SCENARIOS = [
    'task_visit_single',
    'task_visit_single_hidden_flex',
    'task_visit_multi',
    'task_visit_single_seller_followup',
]

def format_prompt(obs, step: int) -> str:
    visible_state = {
        'step': step,
        'phase': obs.phase,
        'status': obs.status,
        'remaining_required_fields': obs.remaining_required_fields,
        'available_tools': obs.available_tools,
        'feedback_summary': obs.feedback_summary,
        'message': obs.message,
        'last_tool_result': obs.last_tool_result,
        'buyer_history': obs.buyer_conversation_history[-4:],
        'seller_history': obs.seller_conversation_history[-4:],
    }

    return (
        'Return exactly one JSON object.\\n'
        'Schema: {"action_type":"assistant_message","assistant_message":"..."} or '
        '{"action_type":"tool_call","tool_name":"...","tool_arguments":{...}}\\n\\n'
        f'Observation:\n{json.dumps(visible_state, ensure_ascii=False, indent=2)}\n'
        'Return JSON only.'
    )

rows = []
for scenario_id in TARGET_SCENARIOS:
    env = FlatmateRlEnvironment()
    obs = env.reset(scenario_id=scenario_id)
    for step in (1, 2):
        payload = expected_policy_action(scenario_id, obs.model_dump())
        if payload is None:
            break
        rows.append(
            {
                'scenario_id': scenario_id,
                'step': step,
                'prompt': format_prompt(obs, step),
                'expected_action': payload,
            }
        )
        obs = env.step(FlatmateRlAction.model_validate(payload))

train_ds = Dataset.from_list(rows)
train_ds[:2]


In [ ]:
def score_completion(example, completion_text: str) -> float:
    try:
        action = json.loads(completion_text)
    except json.JSONDecodeError:
        return -0.25

    step = int(example['step'])
    expected = example['expected_action']

    if step == 1:
        message = str(action.get('assistant_message', '')).lower()
        if action.get('action_type') == 'assistant_message' and 'diet' in message and 'availability' in message:
            return 1.0
        return -0.1

    if step == 2:
        if action.get('action_type') == 'tool_call' and action.get('tool_name') == expected.get('tool_name'):
            return 1.0
        return -0.2

    return 0.0

for row in rows[:2]:
    print(row['scenario_id'], row['step'], score_completion(row, json.dumps(row['expected_action'])))


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')

from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

grpo_args = GRPOConfig(
    output_dir='flatmate_grpo_step2',
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_prompt_length=1024,
    max_completion_length=256,
    num_generations=4,
    logging_steps=1,
    save_steps=25,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

def reward_func(prompts, completions, **kwargs):
    rewards = []
    examples = kwargs['examples']
    for example, completion in zip(examples, completions):
        rewards.append(score_completion(example, completion))
    return rewards

# Starter training block.
# If your installed TRL version expects a slightly different GRPOTrainer signature,
# keep the dataset, reward, and LoRA config from above and adapt only the constructor call.
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=grpo_args,
    train_dataset=train_ds,
    reward_funcs=[reward_func],
    peft_config=lora_config,
)

# trainer.train()
print('GRPO trainer configured for the step-1/step-2 curriculum')
